# Import data into QuickBooks Online

Load the existing CloudFlow CSVs into the configured QuickBooks sandbox in this order: **Chart of Accounts, Customers, then Journal Entries**.

Run cells from top to bottom. `APPLY_IMPORT = False` validates locally without API calls. The configuration below currently has `APPLY_IMPORT = True`, which enables API writes when the execution cell runs. Set it to `True` and rerun the configuration and execution cells when ready to create records. Authentication uses the existing `.env` and `tokens/qbo_tokens.json`; Azure SQL is not needed.

Reusable logic lives in `src/qbo_import.py`.

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "qbo_import.py").exists():
    raise FileNotFoundError("Open this notebook from the project root or notebooks folder.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.qbo_import import load_import_data, QBOClient, import_records

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\kerab\OneDrive\Documentos\Upwork\Portfolio\FP&A Dashboard\quickbooks-pipeline


## Configure the import

In [2]:
DATA_DIR = PROJECT_ROOT / "Data"
ACCOUNT_FILE = "qbo_chart_of_accounts.csv"
CUSTOMER_FILE = "customer_master.csv"
JOURNAL_FILES = [
    "qbo_journal_import_part1.csv",
    "qbo_journal_import_part2.csv",
]
APPLY_IMPORT = True

# These four files are loaded. Set CUSTOMER_FILE = None to skip customers.

## Validate and preview

Validation checks required columns, unique identities, account/customer references, amounts, and balanced journals. Customer_ID maps to QBO DisplayName; populated source attributes are preserved in Notes. QBO assigns its own internal customer Id. MRR values remain descriptive notes and do not create transactions. Churn_Date does not deactivate a customer. A nonempty journal Name must match a Customer_ID.

CSV detail labels map to US API values. The two Other Expense accounts use OtherMiscellaneousExpense to preserve their classification. Nonempty TaxCode, Location, and Class are not supported yet and cause validation to fail.

In [3]:
accounts, customers, journals = load_import_data(
    DATA_DIR, account_file=ACCOUNT_FILE, journal_files=JOURNAL_FILES, customer_file=CUSTOMER_FILE
)

summary = pd.DataFrame([
    {"Entity": "Account", "Records": len(accounts)},
    {"Entity": "Customer", "Records": len(customers)},
    {"Entity": "JournalEntry", "Records": len(journals)},
    {"Entity": "Journal lines", "Records": sum(len(j["Line"]) for j in journals)},
])
display(summary)
print("All source journals are balanced. No API calls made.")

,Entity,Records
0,Account,46
1,Customer,1068
2,JournalEntry,36
3,Journal lines,1577


All source journals are balanced. No API calls made.


In [4]:
display(pd.DataFrame(accounts).head())
display(pd.DataFrame(customers).head())
display(pd.DataFrame([
    {"Journal": j["DocNumber"], "Date": j["TxnDate"], "Lines": len(j["Line"])}
    for j in journals
]))

,Name,AcctNum,AccountType,AccountSubType
0,Checking,1000,Bank,Checking
1,Subscription Revenue - Core Platform,4000,Income,ServiceFeeIncome
2,Subscription Revenue - Premium Add-ons,4010,Income,ServiceFeeIncome
3,Usage-based Revenue,4020,Income,ServiceFeeIncome
4,Professional Services - Implementation,4100,Income,ServiceFeeIncome


,DisplayName,Notes
0,C00001,Customer_ID=C00001; Segment=SMB; Region=North ...
1,C00002,Customer_ID=C00002; Segment=SMB; Region=LATAM;...
2,C00003,Customer_ID=C00003; Segment=SMB; Region=North ...
3,C00004,Customer_ID=C00004; Segment=SMB; Region=North ...
4,C00005,Customer_ID=C00005; Segment=SMB; Region=North ...


,Journal,Date,Lines
0,202309,2023-09-30,43
1,202310,2023-10-31,43
2,202311,2023-11-30,44
3,202312,2023-12-31,43
4,202401,2024-01-31,44
5,202402,2024-02-29,44
6,202403,2024-03-31,43
7,202404,2024-04-30,44
8,202405,2024-05-31,43
9,202406,2024-06-30,44


## Execute

This cell reloads and validates the files, then checks existing QuickBooks records before creating anything. Matching records are reused; conflicts or inactive records stop the import. No existing records are updated or deleted.

Imports are not atomic: successful records remain after a later failure. Rerun unchanged inputs to resume. Avoid concurrent imports. Stable request IDs protect identical retries; after a sandbox reset, old request IDs may still be replayed. API acceptance of locale-specific account types is verified only during import.

In [5]:
import_results = []
if not APPLY_IMPORT:
    print("Validation-only mode. Set APPLY_IMPORT = True above to import into QuickBooks.")
else:
    accounts, customers, journals = load_import_data(
        DATA_DIR, account_file=ACCOUNT_FILE, journal_files=JOURNAL_FILES, customer_file=CUSTOMER_FILE
    )
    client = QBOClient()
    try:
        import_results = import_records(accounts, customers, journals, client)
    finally:
        client.session.close()
    print("Import complete.")

To create: 1113 master records, 36 journals.
Created Account Subscription Revenue - Core Platform: Id=1150040000
Created Account Subscription Revenue - Premium Add-ons: Id=1150040001
Created Account Usage-based Revenue: Id=1150040002
Created Account Professional Services - Implementation: Id=1150040003
Created Account Professional Services - Training & Consulting: Id=1150040004
Created Account Contra Revenue - Discounts & Credits: Id=1150040005
Created Account COGS - Cloud Compute: Id=1150040006
Created Account COGS - Database & Storage: Id=1150040007
Created Account COGS - Network & CDN: Id=1150040008
Created Account COGS - Monitoring & Observability: Id=1150040009
Created Account COGS - External APIs: Id=1150040010
Created Account COGS - Email & SMS: Id=1150040011
Created Account COGS - Customer Support Salaries: Id=1150040012
Created Account COGS - Customer Support Benefits & Payroll Tax: Id=1150040013
Created Account COGS - Customer Support Contractors: Id=1150040014
Created Accoun

In [6]:
if import_results:
    result_df = pd.DataFrame(import_results)
    display(result_df.groupby(["entity", "status"]).size().rename("records").reset_index())
else:
    print("No import results: validation-only mode or execution has not completed.")

,entity,status,records
0,Account,created,45
1,Account,reused,1
2,Customer,created,1068
3,JournalEntry,created,36


After a successful import, run `01_extract_quickbooks_bronze.ipynb` or the pipeline notebook to refresh the analytical data.

Reference: [Intuit API best practices and request IDs](https://blogs.a.intuit.com/2018/09/10/quickbooks-online-api-best-practices/).